# Stage 5: LLM-Based Few-Shot Relation Extraction

## Project
**Comparing LLMs and Fine-Tuned Transformer-Based Models for Biomedical Named Entity Recognition and Relation Extraction for Knowledge Graph Construction**

## Objective
The objective of Stage 5 is to perform **few-shot biomedical relation extraction** on the BioRED dataset using **Qwen2.5-7B-Instruct**.

The model will be provided with biomedical text and predefined entity pairs and will predict the relationship between those entities according to the BioRED relation schema.

This stage reuses the infrastructure developed during Stage 4 wherever possible, including:

- Qwen2.5-7B-Instruct model and tokenizer
- 4-bit quantized model loading
- Few-shot prompting framework
- Structured JSON generation
- JSON parsing and validation
- Batch inference
- Checkpoint saving
- Resume capability
- Evaluation framework

The NER-specific components from Stage 4 will be adapted for relation classification.

## Stage 5 Pipeline

1. Load and inspect the prepared BioRED relation extraction data
2. Define the relation extraction schema
3. Prepare few-shot relation examples
4. Construct the relation extraction prompt
5. Load Qwen2.5-7B-Instruct
6. Perform and validate single-example inference
7. Run development-set inference
8. Evaluate development performance
9. Freeze the final few-shot prompt
10. Run test-set inference and evaluation
11. Compare LLM-based RE with fine-tuned PubMedBERT RE

In [1]:
# Stage 5.1A - Mount Google Drive and Set Project Path

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Capstone_Project")

print("Project directory:", PROJECT_DIR)
print("Exists:", PROJECT_DIR.exists())

Mounted at /content/drive
Project directory: /content/drive/MyDrive/Capstone_Project
Exists: True


In [2]:
# Stage 5.1B - Define Stage 5 Paths and Inspect Saved RE Outputs

from pathlib import Path

STAGE_3_2_DIR = PROJECT_DIR / "Stage_3_2_Outputs"
STAGE_5_DIR = PROJECT_DIR / "Stage_5_LLM_RE"

STAGE_5_DIR.mkdir(parents=True, exist_ok=True)

print("Stage 3.2 outputs directory:", STAGE_3_2_DIR)
print("Exists:", STAGE_3_2_DIR.exists())

print("\nStage 5 output directory:", STAGE_5_DIR)
print("Exists:", STAGE_5_DIR.exists())

print("\nFiles available in Stage_3_2_Outputs:")

if STAGE_3_2_DIR.exists():
    for file_path in sorted(STAGE_3_2_DIR.iterdir()):
        print("-", file_path.name)
else:
    print("Stage_3_2_Outputs directory not found.")

Stage 3.2 outputs directory: /content/drive/MyDrive/Capstone_Project/Stage_3_2_Outputs
Exists: True

Stage 5 output directory: /content/drive/MyDrive/Capstone_Project/Stage_5_LLM_RE
Exists: True

Files available in Stage_3_2_Outputs:
- biored_prepared_relation_data.pkl
- biored_relation_label_mappings.pkl


In [3]:
# Stage 5.1C - Load Prepared Relation Data and Label Mappings

import pickle

RELATION_DATA_PATH = STAGE_3_2_DIR / "biored_prepared_relation_data.pkl"
LABEL_MAPPING_PATH = STAGE_3_2_DIR / "biored_relation_label_mappings.pkl"

with open(RELATION_DATA_PATH, "rb") as f:
    relation_data = pickle.load(f)

with open(LABEL_MAPPING_PATH, "rb") as f:
    relation_label_mappings = pickle.load(f)

print("Relation data type:", type(relation_data))
print("Relation data keys:", relation_data.keys())

print("\nLabel mappings type:", type(relation_label_mappings))
print("Label mapping keys:", relation_label_mappings.keys())

Relation data type: <class 'dict'>
Relation data keys: dict_keys(['train', 'development', 'test'])

Label mappings type: <class 'dict'>
Label mapping keys: dict_keys(['label_to_id', 'id_to_label'])


In [4]:
print("\nDataset sizes:")

for split_name, split_data in relation_data.items():
    print(f"{split_name}: {len(split_data)} examples")

print("\nLabel mappings:")
print(relation_label_mappings)

print("\nFirst training example:")
print(relation_data["train"][0])


Dataset sizes:
train: 8356 examples
development: 2324 examples
test: 2326 examples

Label mappings:
{'label_to_id': {'Association': 0, 'Bind': 1, 'Comparison': 2, 'Conversion': 3, 'Cotreatment': 4, 'Drug_Interaction': 5, 'Negative_Correlation': 6, 'Positive_Correlation': 7, 'No_Relation': 8}, 'id_to_label': {0: 'Association', 1: 'Bind', 2: 'Comparison', 3: 'Conversion', 4: 'Cotreatment', 5: 'Drug_Interaction', 6: 'Negative_Correlation', 7: 'Positive_Correlation', 8: 'No_Relation'}}

First training example:
{'document_id': '26115410', 'text': 'Mechanisms Underlying Latent Disease Risk Associated with Early-Life Arsenic Exposure: Current Research Trends and Scientific Gaps. BACKGROUND: Millions of individuals worldwide, particularly those living in rural and developing areas, are exposed to harmful levels of inorganic arsenic (iAs) in their drinking water. Inorganic As exposure during key developmental periods is associated with a variety of adverse health effects including those that a

In [5]:
# Stage 5.1D - Validate Prepared Relation Extraction Data

required_fields = {
    "document_id",
    "text",
    "entity1_text",
    "entity1_type",
    "entity2_text",
    "entity2_type",
    "relation_label",
    "marked_text",
}

valid_labels = set(relation_label_mappings["label_to_id"].keys())

for split_name, split_data in relation_data.items():

    missing_field_examples = 0
    invalid_label_examples = 0
    missing_marked_text_examples = 0

    for example in split_data:

        # Check required fields
        if not required_fields.issubset(example.keys()):
            missing_field_examples += 1

        # Check relation label
        if example.get("relation_label") not in valid_labels:
            invalid_label_examples += 1

        # Check marked text
        if not example.get("marked_text"):
            missing_marked_text_examples += 1

    print(f"\n{split_name.upper()}")
    print("Total examples:", len(split_data))
    print("Missing required fields:", missing_field_examples)
    print("Invalid relation labels:", invalid_label_examples)
    print("Missing marked text:", missing_marked_text_examples)


TRAIN
Total examples: 8356
Missing required fields: 0
Invalid relation labels: 0
Missing marked text: 0

DEVELOPMENT
Total examples: 2324
Missing required fields: 0
Invalid relation labels: 0
Missing marked text: 0

TEST
Total examples: 2326
Missing required fields: 0
Invalid relation labels: 0
Missing marked text: 0


In [4]:
# Stage 5.2A - Define Relation Labels and Structured Output Schema

RELATION_LABELS = list(
    relation_label_mappings["label_to_id"].keys()
)

RELATION_OUTPUT_SCHEMA = {
    "relation": "<one valid BioRED relation label>"
}

print("Relation labels:")
for label in RELATION_LABELS:
    print("-", label)

print("\nExpected JSON output:")
print(RELATION_OUTPUT_SCHEMA)

Relation labels:
- Association
- Bind
- Comparison
- Conversion
- Cotreatment
- Drug_Interaction
- Negative_Correlation
- Positive_Correlation
- No_Relation

Expected JSON output:
{'relation': '<one valid BioRED relation label>'}


In [5]:
# Stage 5.2B - Create Relation Output Validation Function

def validate_relation_output(parsed_output):
    """
    Validate a parsed LLM relation extraction output.

    Expected format:
        {"relation": "<valid BioRED relation label>"}

    Returns:
        (is_valid, relation, error_message)
    """

    # Output must be a dictionary
    if not isinstance(parsed_output, dict):
        return False, None, "Output is not a dictionary."

    # Required field must exist
    if "relation" not in parsed_output:
        return False, None, "Missing 'relation' field."

    relation = parsed_output["relation"]

    # Relation must be a string
    if not isinstance(relation, str):
        return False, None, "'relation' must be a string."

    # Remove accidental surrounding whitespace
    relation = relation.strip()

    # Relation must belong to the BioRED schema
    if relation not in RELATION_LABELS:
        return False, None, f"Invalid relation label: {relation}"

    return True, relation, None

In [6]:
valid_example = {
    "relation": "Positive_Correlation"
}

invalid_example = {
    "relation": "Causes"
}

print("Valid example:")
print(validate_relation_output(valid_example))

print("\nInvalid example:")
print(validate_relation_output(invalid_example))

Valid example:
(True, 'Positive_Correlation', None)

Invalid example:
(False, None, 'Invalid relation label: Causes')


In [7]:
# Stage 5.2C - Create JSON Parsing Function

import json
import re

def parse_relation_json(raw_output):
    """
    Parse JSON from the raw LLM output.

    Returns:
        (parsed_output, error_message)
    """

    if not isinstance(raw_output, str):
        return None, "Raw output is not a string."

    text = raw_output.strip()

    # Remove Markdown code fences if present
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)

    # First try parsing the full output directly
    try:
        return json.loads(text), None
    except json.JSONDecodeError:
        pass

    # If extra text is present, try extracting the first JSON object
    match = re.search(r"\{.*?\}", text, flags=re.DOTALL)

    if match:
        try:
            return json.loads(match.group()), None
        except json.JSONDecodeError as error:
            return None, f"Invalid JSON: {error}"

    return None, "No JSON object found."

In [8]:
test_outputs = [
    '{"relation": "Positive_Correlation"}',

    '''```json
    {"relation": "Bind"}
    ```''',

    'The answer is {"relation": "Association"}',

    'relation = Positive_Correlation'
]

for i, output in enumerate(test_outputs, start=1):

    parsed, error = parse_relation_json(output)

    print(f"\nTest {i}")
    print("Parsed:", parsed)
    print("Error:", error)


Test 1
Parsed: {'relation': 'Positive_Correlation'}
Error: None

Test 2
Parsed: {'relation': 'Bind'}
Error: None

Test 3
Parsed: {'relation': 'Association'}
Error: None

Test 4
Parsed: None
Error: No JSON object found.


In [9]:
# Stage 5.2D - Create Combined Parsing and Validation Function

def process_relation_output(raw_output):
    """
    Parse and validate a raw LLM relation extraction output.

    Returns:
        {
            "valid_json": bool,
            "valid_relation": bool,
            "relation": str or None,
            "error": str or None
        }
    """

    parsed_output, parse_error = parse_relation_json(raw_output)

    if parsed_output is None:
        return {
            "valid_json": False,
            "valid_relation": False,
            "relation": None,
            "error": parse_error
        }

    is_valid, relation, validation_error = validate_relation_output(
        parsed_output
    )

    if not is_valid:
        return {
            "valid_json": True,
            "valid_relation": False,
            "relation": None,
            "error": validation_error
        }

    return {
        "valid_json": True,
        "valid_relation": True,
        "relation": relation,
        "error": None
    }

In [10]:
test_outputs = [
    '{"relation": "Positive_Correlation"}',
    '{"relation": "Causes"}',
    'not valid json'
]

for i, output in enumerate(test_outputs, start=1):
    result = process_relation_output(output)

    print(f"\nTest {i}")
    print(result)


Test 1
{'valid_json': True, 'valid_relation': True, 'relation': 'Positive_Correlation', 'error': None}

Test 2
{'valid_json': True, 'valid_relation': False, 'relation': None, 'error': 'Invalid relation label: Causes'}

Test 3
{'valid_json': False, 'valid_relation': False, 'relation': None, 'error': 'No JSON object found.'}


In [14]:
# Stage 5.3A - Inspect Training Relation Label Distribution

from collections import Counter

train_label_counts = Counter(
    example["relation_label"]
    for example in relation_data["train"]
)

print("Training relation label distribution:\n")

for label in RELATION_LABELS:
    print(f"{label}: {train_label_counts[label]}")

total_train_examples = len(relation_data["train"])

print("\nTraining relation label percentages:\n")

for label in RELATION_LABELS:
    count = train_label_counts[label]
    percentage = (count / total_train_examples) * 100

    print(
        f"{label}: "
        f"{count} examples "
        f"({percentage:.2f}%)"
    )

Training relation label distribution:

Association: 2192
Bind: 61
Comparison: 28
Conversion: 3
Cotreatment: 31
Drug_Interaction: 11
Negative_Correlation: 763
Positive_Correlation: 1089
No_Relation: 4178

Training relation label percentages:

Association: 2192 examples (26.23%)
Bind: 61 examples (0.73%)
Comparison: 28 examples (0.34%)
Conversion: 3 examples (0.04%)
Cotreatment: 31 examples (0.37%)
Drug_Interaction: 11 examples (0.13%)
Negative_Correlation: 763 examples (9.13%)
Positive_Correlation: 1089 examples (13.03%)
No_Relation: 4178 examples (50.00%)


In [15]:
# Stage 5.3B - Inspect Candidate Few-Shot Example Lengths

candidates_by_label = {}

for label in RELATION_LABELS:

    label_examples = [
        example
        for example in relation_data["train"]
        if example["relation_label"] == label
    ]

    # Sort by marked-text character length
    label_examples = sorted(
        label_examples,
        key=lambda example: len(example["marked_text"])
    )

    candidates_by_label[label] = label_examples

    print(f"\n{label}")
    print("Available examples:", len(label_examples))

    for i, example in enumerate(label_examples[:3], start=1):
        print(
            f"Candidate {i}: "
            f"Document {example['document_id']} | "
            f"{len(example['marked_text'])} characters | "
            f"{example['entity1_text']} -> {example['entity2_text']}"
        )


Association
Available examples: 2192
Candidate 1: Document 7468724 | 425 characters | terbutaline -> corticosteroid
Candidate 2: Document 7468724 | 449 characters | Cardiovascular complications -> terbutaline
Candidate 3: Document 18262054 | 575 characters | Y245X -> G246X

Bind
Available examples: 61
Candidate 1: Document 10661407 | 1081 characters | Langerin -> mannose
Candidate 2: Document 19463742 | 1098 characters | melanocortin -> Melanocortin-4 receptor
Candidate 3: Document 22051099 | 1186 characters | CXCR1 -> interleukin 8

Comparison
Available examples: 28
Candidate 1: Document 9746003 | 835 characters | carbamazepine -> vigabatrin
Candidate 2: Document 18631865 | 1069 characters | calcineurin inhibitors -> sirolimus
Candidate 3: Document 16920333 | 1388 characters | carbamazepine -> oxcarbazepine

Conversion
Available examples: 3
Candidate 1: Document 17391797 | 1589 characters | phosphatidylethanolamine -> phosphatidylcholine
Candidate 2: Document 16506214 | 1780 characte

In [16]:
# Stage 5.3C - Inspect Few-Shot Candidate Examples

NUM_CANDIDATES_TO_INSPECT = 3

for label in RELATION_LABELS:

    print("\n" + "=" * 100)
    print(f"RELATION LABEL: {label}")
    print("=" * 100)

    candidates = candidates_by_label[label][:NUM_CANDIDATES_TO_INSPECT]

    for i, example in enumerate(candidates, start=1):

        print(f"\nCandidate {i}")
        print("Document ID:", example["document_id"])
        print(
            f"Entity 1: {example['entity1_text']} "
            f"({example['entity1_type']})"
        )
        print(
            f"Entity 2: {example['entity2_text']} "
            f"({example['entity2_type']})"
        )
        print("Gold relation:", example["relation_label"])
        print("\nMarked text:")
        print(example["marked_text"])
        print("-" * 100)


RELATION LABEL: Association

Candidate 1
Document ID: 7468724
Entity 1: terbutaline (ChemicalEntity)
Entity 2: corticosteroid (ChemicalEntity)
Gold relation: Association

Marked text:
Cardiovascular complications associated with @ChemicalEntity$ terbutaline @/ChemicalEntity$ treatment for preterm labor. Severe cardiovascular complications occurred in eight of 160 patients treated with terbutaline for preterm labor. Associated #ChemicalEntity$ corticosteroid #/ChemicalEntity$ therapy and twin gestations appear to be predisposing factors. Potential mechanisms of the pathophysiology are briefly discussed.
----------------------------------------------------------------------------------------------------

Candidate 2
Document ID: 7468724
Entity 1: Cardiovascular complications (DiseaseOrPhenotypicFeature)
Entity 2: terbutaline (ChemicalEntity)
Gold relation: Association

Marked text:
@DiseaseOrPhenotypicFeature$ Cardiovascular complications @/DiseaseOrPhenotypicFeature$ associated with #C

In [12]:
# Stage 5.3D - Select and Freeze Few-Shot Demonstrations

FEW_SHOT_SELECTION = [
    {
        "relation_label": "Association",
        "document_id": "7468724",
        "entity1_text": "Cardiovascular complications",
        "entity2_text": "terbutaline",
    },
    {
        "relation_label": "Bind",
        "document_id": "10661407",
        "entity1_text": "Langerin",
        "entity2_text": "mannose",
    },
    {
        "relation_label": "Comparison",
        "document_id": "16920333",
        "entity1_text": "carbamazepine",
        "entity2_text": "oxcarbazepine",
    },
    {
        "relation_label": "Conversion",
        "document_id": "17391797",
        "entity1_text": "phosphatidylethanolamine",
        "entity2_text": "phosphatidylcholine",
    },
    {
        "relation_label": "Cotreatment",
        "document_id": "9746003",
        "entity1_text": "sodium valproate",
        "entity2_text": "lamotrigine",
    },
    {
        "relation_label": "Drug_Interaction",
        "document_id": "19642243",
        "entity1_text": "tenofovir",
        "entity2_text": "vancomycin",
    },
    {
        "relation_label": "Negative_Correlation",
        "document_id": "7468724",
        "entity1_text": "terbutaline",
        "entity2_text": "preterm labor",
    },
    {
        "relation_label": "Positive_Correlation",
        "document_id": "15602202",
        "entity1_text": "interstitial nephritis",
        "entity2_text": "azithromycin",
    },
    {
        "relation_label": "No_Relation",
        "document_id": "18262054",
        "entity1_text": "G246X",
        "entity2_text": "human",
    },
]
def find_training_example(selection):
    """
    Find one exact training example matching the frozen
    few-shot selection specification.
    """

    matches = [
        example
        for example in relation_data["train"]
        if (
            example["document_id"] == selection["document_id"]
            and example["relation_label"] == selection["relation_label"]
            and example["entity1_text"] == selection["entity1_text"]
            and example["entity2_text"] == selection["entity2_text"]
        )
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected exactly one match for {selection}, "
            f"but found {len(matches)}."
        )

    return matches[0]


few_shot_examples = [
    find_training_example(selection)
    for selection in FEW_SHOT_SELECTION
]

print("Number of frozen few-shot examples:", len(few_shot_examples))

for i, example in enumerate(few_shot_examples, start=1):
    print(
        f"{i}. {example['relation_label']} | "
        f"{example['entity1_text']} -> {example['entity2_text']} | "
        f"Document {example['document_id']}"
    )

Number of frozen few-shot examples: 9
1. Association | Cardiovascular complications -> terbutaline | Document 7468724
2. Bind | Langerin -> mannose | Document 10661407
3. Comparison | carbamazepine -> oxcarbazepine | Document 16920333
4. Conversion | phosphatidylethanolamine -> phosphatidylcholine | Document 17391797
5. Cotreatment | sodium valproate -> lamotrigine | Document 9746003
6. Drug_Interaction | tenofovir -> vancomycin | Document 19642243
7. Negative_Correlation | terbutaline -> preterm labor | Document 7468724
8. Positive_Correlation | interstitial nephritis -> azithromycin | Document 15602202
9. No_Relation | G246X -> human | Document 18262054


In [13]:
# Stage 5.3E - Save Frozen Few-Shot Demonstrations

FEW_SHOT_SELECTION_PATH = STAGE_5_DIR / "few_shot_selection.pkl"
FEW_SHOT_EXAMPLES_PATH = STAGE_5_DIR / "few_shot_examples.pkl"

with open(FEW_SHOT_SELECTION_PATH, "wb") as f:
    pickle.dump(FEW_SHOT_SELECTION, f)

with open(FEW_SHOT_EXAMPLES_PATH, "wb") as f:
    pickle.dump(few_shot_examples, f)

print("Saved few-shot selection to:")
print(FEW_SHOT_SELECTION_PATH)

print("\nSaved few-shot examples to:")
print(FEW_SHOT_EXAMPLES_PATH)

print("\nNumber of saved examples:", len(few_shot_examples))

Saved few-shot selection to:
/content/drive/MyDrive/Capstone_Project/Stage_5_LLM_RE/few_shot_selection.pkl

Saved few-shot examples to:
/content/drive/MyDrive/Capstone_Project/Stage_5_LLM_RE/few_shot_examples.pkl

Number of saved examples: 9


In [14]:
# Stage 5.4A - Create Reusable Relation Example Formatter

def format_relation_example(example, include_answer=False):
    """
    Format one BioRED relation example for the LLM prompt.

    Entity 1 is marked using @...$ markers.
    Entity 2 is marked using #...$ markers.

    Args:
        example: BioRED relation example dictionary.
        include_answer: Whether to include the gold relation label.

    Returns:
        Formatted string.
    """

    formatted = (
        f"Document:\n"
        f"{example['marked_text']}\n\n"
        f"Entity 1: {example['entity1_text']}\n"
        f"Entity 1 type: {example['entity1_type']}\n"
        f"Entity 2: {example['entity2_text']}\n"
        f"Entity 2 type: {example['entity2_type']}"
    )

    if include_answer:
        formatted += (
            f'\n\nOutput:\n'
            f'{{"relation": "{example["relation_label"]}"}}'
        )

    return formatted

In [16]:
print(
    format_relation_example(
        few_shot_examples[0],
        include_answer=True
    )
)

Document:
@DiseaseOrPhenotypicFeature$ Cardiovascular complications @/DiseaseOrPhenotypicFeature$ associated with #ChemicalEntity$ terbutaline #/ChemicalEntity$ treatment for preterm labor. Severe cardiovascular complications occurred in eight of 160 patients treated with terbutaline for preterm labor. Associated corticosteroid therapy and twin gestations appear to be predisposing factors. Potential mechanisms of the pathophysiology are briefly discussed.

Entity 1: Cardiovascular complications
Entity 1 type: DiseaseOrPhenotypicFeature
Entity 2: terbutaline
Entity 2 type: ChemicalEntity

Output:
{"relation": "Association"}


In [51]:
# Stage 5.4B - Define Fixed Relation Extraction Instructions

RELATION_INSTRUCTIONS = f"""
You are performing biomedical relation extraction on BioRED data.

Your task is to classify the relationship between Entity 1 and Entity 2.

The document contains special markers:
- Entity 1 is marked using @...$ and @/...$
- Entity 2 is marked using #...$ and #/...$

You MUST choose exactly one relation label from this closed label set:

{", ".join(RELATION_LABELS)}

Rules:
1. Predict only the relationship between the specified Entity 1 and Entity 2.
2. Use the biomedical context in the document.
3. Do not predict relationships involving other entities.
4. You must use ONLY one of the labels listed above.
5. Do not invent alternative labels such as Cause, Causes, Causal, Related_To, Interaction, or None.
6. If the relationship is causal or indicates increased occurrence/risk, map it to the most appropriate BioRED label from the allowed set.
7. If no annotated relationship exists between Entity 1 and Entity 2, return No_Relation.
8. Output valid JSON only.
9. Do not include explanations, reasoning, Markdown, or additional text.

Required output format:
{{"relation": "<one allowed BioRED relation label>"}}
""".strip()

In [17]:
# Stage 5.4C - Build Complete Few-Shot Relation Extraction Prompt

def build_relation_prompt(example, few_shot_examples, instructions):
    """
    Build the complete few-shot relation extraction prompt.

    Args:
        example: Target BioRED relation example to classify.
        few_shot_examples: Frozen few-shot demonstration examples.
        instructions: Fixed relation extraction instruction block.

    Returns:
        Complete prompt string.
    """

    prompt_parts = [instructions]

    prompt_parts.append(
        "\n\nFew-shot examples:"
    )

    for i, demo in enumerate(few_shot_examples, start=1):

        formatted_demo = format_relation_example(
            demo,
            include_answer=True
        )

        prompt_parts.append(
            f"\n\nExample {i}\n"
            f"{formatted_demo}"
        )

    target_example = format_relation_example(
        example,
        include_answer=False
    )

    prompt_parts.append(
        "\n\nNow classify the following example:\n"
        f"{target_example}\n\n"
        "Output:"
    )

    return "".join(prompt_parts)

In [18]:
# Find one non-demonstration training example for prompt testing

few_shot_keys = {
    (
        example["document_id"],
        example["entity1_annotation_id"],
        example["entity2_annotation_id"],
        example["relation_label"]
    )
    for example in few_shot_examples
}

prompt_test_example = next(
    example
    for example in relation_data["train"]
    if (
        example["document_id"],
        example["entity1_annotation_id"],
        example["entity2_annotation_id"],
        example["relation_label"]
    ) not in few_shot_keys
)

test_prompt = build_relation_prompt(
    prompt_test_example,
    few_shot_examples,
    RELATION_INSTRUCTIONS
)

print(test_prompt)

You are performing biomedical relation extraction on BioRED data.

Your task is to classify the relationship between Entity 1 and Entity 2.

The document contains special markers:
- Entity 1 is marked using @...$ and @/...$
- Entity 2 is marked using #...$ and #/...$

Choose exactly one relation label from the following list:

Association, Bind, Comparison, Conversion, Cotreatment, Drug_Interaction, Negative_Correlation, Positive_Correlation, No_Relation

Rules:
1. Predict only the relationship between the specified Entity 1 and Entity 2.
2. Use the biomedical context in the document.
3. Do not predict relationships involving other entities.
4. Return exactly one valid relation label.
5. If no annotated relationship exists between Entity 1 and Entity 2, return No_Relation.
6. Output valid JSON only.
7. Do not include explanations, reasoning, Markdown, or additional text.

Required output format:
{"relation": "<relation_label>"}

Few-shot examples:

Example 1
Document:
@DiseaseOrPhenoty

In [25]:
# Stage 5.4D - Inspect Few-Shot Prompt Size

prompt_character_count = len(test_prompt)
prompt_word_count = len(test_prompt.split())

print("Prompt characters:", prompt_character_count)
print("Approximate prompt words:", prompt_word_count)

print("\nFew-shot demonstrations:", len(few_shot_examples))
print("Target document ID:", prompt_test_example["document_id"])

Prompt characters: 11877
Approximate prompt words: 1514

Few-shot demonstrations: 9
Target document ID: 26115410


In [19]:
# Stage 5.5A - Install Required Libraries

!pip install -q transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 66.8 MB/s eta 0:00:00


In [20]:
# Stage 5.5B - Import Model Libraries and Define Model Configuration

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

print("Model:", MODEL_NAME)
print("4-bit quantization configured.")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Model: Qwen/Qwen2.5-7B-Instruct
4-bit quantization configured.
CUDA available: True
GPU: NVIDIA L4


In [21]:
# Stage 5.5C - Load Qwen Tokenizer and 4-Bit Model

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True
)

model.eval()

print("Tokenizer loaded.")
print("Model loaded.")
print("Model device:", model.device)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Tokenizer loaded.
Model loaded.
Model device: cuda:0


In [29]:
print("\nTokenizer vocabulary size:", len(tokenizer))
print("Model dtype:", model.dtype)


Tokenizer vocabulary size: 151665
Model dtype: torch.bfloat16


In [30]:
# Stage 5.5D - Measure Exact Prompt Token Length

prompt_tokens = tokenizer(
    test_prompt,
    add_special_tokens=False,
    return_tensors=None
)["input_ids"]

prompt_token_count = len(prompt_tokens)

print("Exact prompt token count:", prompt_token_count)

print(
    "Model maximum context length:",
    tokenizer.model_max_length
)

Exact prompt token count: 2840
Model maximum context length: 131072


In [31]:
context_usage_percent = (
    prompt_token_count / tokenizer.model_max_length
) * 100

print(
    f"Approximate context usage: "
    f"{context_usage_percent:.2f}%"
)

Approximate context usage: 2.17%


In [22]:
# Stage 5.5E - Create Reusable Qwen Generation Function

def generate_relation_output(
    prompt,
    max_new_tokens=64
):
    """
    Generate a relation extraction response using Qwen2.5-7B-Instruct.

    Args:
        prompt: Complete few-shot relation extraction prompt.
        max_new_tokens: Maximum number of tokens to generate.

    Returns:
        Raw generated text.
    """

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[
        0,
        inputs["input_ids"].shape[1]:
    ]

    raw_output = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return raw_output

In [33]:
# Stage 5.5F - Test Single Relation Extraction Generation

raw_output = generate_relation_output(
    test_prompt
)

print("Raw model output:")
print(raw_output)

print("\nGold relation:")
print(prompt_test_example["relation_label"])

Raw model output:
{"relation": "Positive_Correlation"}

Gold relation:
Positive_Correlation


In [34]:
# Stage 5.5G - Process and Validate Single Model Prediction

processed_output = process_relation_output(raw_output)

print("Raw output:")
print(raw_output)

print("\nProcessed output:")
print(processed_output)

print("\nGold relation:")
print(prompt_test_example["relation_label"])

print(
    "\nPrediction correct:",
    processed_output["relation"] == prompt_test_example["relation_label"]
)

Raw output:
{"relation": "Positive_Correlation"}

Processed output:
{'valid_json': True, 'valid_relation': True, 'relation': 'Positive_Correlation', 'error': None}

Gold relation:
Positive_Correlation

Prediction correct: True


In [23]:
# Stage 5.6A - Create Complete Single-Example Relation Inference Function

def run_relation_inference(example):
    """
    Run the complete Stage 5 relation extraction pipeline
    for one BioRED relation example.

    Returns:
        Dictionary containing prediction details.
    """

    prompt = build_relation_prompt(
        example,
        few_shot_examples,
        RELATION_INSTRUCTIONS
    )

    raw_output = generate_relation_output(prompt)

    processed_output = process_relation_output(raw_output)

    result = {
        "document_id": example["document_id"],
        "entity1_text": example["entity1_text"],
        "entity1_type": example["entity1_type"],
        "entity2_text": example["entity2_text"],
        "entity2_type": example["entity2_type"],
        "gold_relation": example["relation_label"],
        "predicted_relation": processed_output["relation"],
        "valid_json": processed_output["valid_json"],
        "valid_relation": processed_output["valid_relation"],
        "raw_output": raw_output,
        "error": processed_output["error"]
    }

    return result

In [24]:
single_result = run_relation_inference(
    prompt_test_example
)

for key, value in single_result.items():
    print(f"{key}: {value}")

document_id: 26115410
entity1_text: inorganic arsenic
entity1_type: ChemicalEntity
entity2_text: cancer
entity2_type: DiseaseOrPhenotypicFeature
gold_relation: Positive_Correlation
predicted_relation: Positive_Correlation
valid_json: True
valid_relation: True
raw_output: {"relation": "Positive_Correlation"}
error: None


In [25]:
# Stage 5.6B - Define Development Checkpoint and Resume Paths

DEV_CHECKPOINT_PATH = STAGE_5_DIR / "development_predictions_checkpoint.pkl"
DEV_FINAL_PATH = STAGE_5_DIR / "development_predictions_final.pkl"

print("Development checkpoint path:")
print(DEV_CHECKPOINT_PATH)

print("\nDevelopment final results path:")
print(DEV_FINAL_PATH)

def load_existing_predictions(checkpoint_path):
    """
    Load existing predictions if a checkpoint exists.
    Otherwise return an empty list.
    """

    if checkpoint_path.exists():
        with open(checkpoint_path, "rb") as f:
            predictions = pickle.load(f)

        print(
            f"Existing checkpoint found: "
            f"{len(predictions)} predictions loaded."
        )

        return predictions

    print("No existing checkpoint found.")
    return []

Development checkpoint path:
/content/drive/MyDrive/Capstone_Project/Stage_5_LLM_RE/development_predictions_checkpoint.pkl

Development final results path:
/content/drive/MyDrive/Capstone_Project/Stage_5_LLM_RE/development_predictions_final.pkl


In [36]:
development_predictions = load_existing_predictions(
    DEV_CHECKPOINT_PATH
)

print(
    "\nPredictions currently available:",
    len(development_predictions)
)

No existing checkpoint found.

Predictions currently available: 0


In [27]:
# Stage 5.6C - Create Resumable Development Inference Loop

def run_development_inference(
    examples,
    checkpoint_path,
    existing_predictions=None,
    max_examples=None,
    checkpoint_every=5
):
    """
    Run resumable relation extraction inference on development examples.

    Args:
        examples: Development examples.
        checkpoint_path: Path used to save intermediate predictions.
        existing_predictions: Previously completed predictions, if any.
        max_examples: Optional limit for testing.
        checkpoint_every: Save checkpoint after this many new predictions.

    Returns:
        List of prediction dictionaries.
    """

    predictions = (
        list(existing_predictions)
        if existing_predictions is not None
        else []
    )

    start_index = len(predictions)

    if max_examples is None:
        end_index = len(examples)
    else:
        end_index = min(start_index + max_examples, len(examples))

    print(f"Starting from example index: {start_index}")
    print(f"Stopping at example index: {end_index}")
    print(f"Already completed: {len(predictions)}")

    for index in range(start_index, end_index):

        example = examples[index]

        try:
            result = run_relation_inference(example)

        except Exception as error:
            result = {
                "document_id": example["document_id"],
                "entity1_text": example["entity1_text"],
                "entity1_type": example["entity1_type"],
                "entity2_text": example["entity2_text"],
                "entity2_type": example["entity2_type"],
                "gold_relation": example["relation_label"],
                "predicted_relation": None,
                "valid_json": False,
                "valid_relation": False,
                "raw_output": None,
                "error": str(error)
            }

        predictions.append(result)

        completed = len(predictions)

        print(
            f"{completed}/{len(examples)} | "
            f"Gold: {result['gold_relation']} | "
            f"Predicted: {result['predicted_relation']}"
        )

        if completed % checkpoint_every == 0:
            with open(checkpoint_path, "wb") as f:
                pickle.dump(predictions, f)

            print(f"Checkpoint saved at {completed} predictions.")

    # Save once more at the end
    with open(checkpoint_path, "wb") as f:
        pickle.dump(predictions, f)

    print("\nInference batch complete.")
    print("Total predictions saved:", len(predictions))

    return predictions

In [35]:
# Stage 5.6F - Reset Development Checkpoint After Prompt Update

if DEV_CHECKPOINT_PATH.exists():
    DEV_CHECKPOINT_PATH.unlink()
    print("Old development checkpoint deleted.")
else:
    print("No checkpoint found.")

development_predictions = []

Old development checkpoint deleted.


In [37]:
development_predictions = run_development_inference(
    examples=relation_data["development"],
    checkpoint_path=DEV_CHECKPOINT_PATH,
    existing_predictions=development_predictions,
    max_examples=5,
    checkpoint_every=5
)

Starting from example index: 0
Stopping at example index: 5
Already completed: 0
1/2324 | Gold: Positive_Correlation | Predicted: Association
2/2324 | Gold: Association | Predicted: Negative_Correlation
3/2324 | Gold: Positive_Correlation | Predicted: Association
4/2324 | Gold: Positive_Correlation | Predicted: Association
5/2324 | Gold: Association | Predicted: Association
Checkpoint saved at 5 predictions.

Inference batch complete.
Total predictions saved: 5


In [38]:
# Stage 5.6G - Verify Development Checkpoint Resume

reloaded_development_predictions = load_existing_predictions(
    DEV_CHECKPOINT_PATH
)

print(
    "\nReloaded prediction count:",
    len(reloaded_development_predictions)
)

if len(reloaded_development_predictions) == 5:
    print("Checkpoint resume verification successful.")
else:
    print("Unexpected checkpoint size.")

Existing checkpoint found: 5 predictions loaded.

Reloaded prediction count: 5
Checkpoint resume verification successful.


In [39]:
# Stage 5.6H - Run Full Development Inference with Checkpointing

development_predictions = run_development_inference(
    examples=relation_data["development"],
    checkpoint_path=DEV_CHECKPOINT_PATH,
    existing_predictions=reloaded_development_predictions,
    max_examples=None,
    checkpoint_every=25
)

Starting from example index: 5
Stopping at example index: 2324
Already completed: 5
6/2324 | Gold: Negative_Correlation | Predicted: Association
7/2324 | Gold: Negative_Correlation | Predicted: None
8/2324 | Gold: Negative_Correlation | Predicted: No_Relation
9/2324 | Gold: Association | Predicted: Bind
10/2324 | Gold: Negative_Correlation | Predicted: Negative_Correlation
11/2324 | Gold: Positive_Correlation | Predicted: Association
12/2324 | Gold: Association | Predicted: Association
13/2324 | Gold: Positive_Correlation | Predicted: Positive_Correlation
14/2324 | Gold: Association | Predicted: Positive_Correlation
15/2324 | Gold: Association | Predicted: Positive_Correlation
16/2324 | Gold: Association | Predicted: No_Relation
17/2324 | Gold: Association | Predicted: Positive_Correlation
18/2324 | Gold: Association | Predicted: Positive_Correlation
19/2324 | Gold: Association | Predicted: Positive_Correlation
20/2324 | Gold: Association | Predicted: Positive_Correlation
21/2324 | Gol

In [40]:
# Stage 5.6I - Save Final Development Predictions

with open(DEV_FINAL_PATH, "wb") as f:
    pickle.dump(development_predictions, f)

print("Final development predictions saved.")
print("Total predictions:", len(development_predictions))
print("Saved to:", DEV_FINAL_PATH)

Final development predictions saved.
Total predictions: 2324
Saved to: /content/drive/MyDrive/Capstone_Project/Stage_5_LLM_RE/development_predictions_final.pkl


In [41]:
# Stage 5.7A - Validate Completed Development Predictions

total_predictions = len(development_predictions)

valid_json_count = sum(
    result["valid_json"]
    for result in development_predictions
)

valid_relation_count = sum(
    result["valid_relation"]
    for result in development_predictions
)

none_prediction_count = sum(
    result["predicted_relation"] is None
    for result in development_predictions
)

print("Total development predictions:", total_predictions)

print(
    "Valid JSON:",
    f"{valid_json_count}/{total_predictions}"
)

print(
    "Valid relation labels:",
    f"{valid_relation_count}/{total_predictions}"
)

print(
    "Invalid / None predictions:",
    none_prediction_count
)

Total development predictions: 2324
Valid JSON: 2324/2324
Valid relation labels: 2300/2324
Invalid / None predictions: 24


In [42]:
# Stage 5.7B - Create Reusable Relation Evaluation Function

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

def evaluate_relation_predictions(predictions):
    """
    Evaluate BioRED relation extraction predictions.
    Invalid/None predictions are counted as incorrect.
    """

    y_true = [
        result["gold_relation"]
        for result in predictions
    ]

    y_pred = [
        result["predicted_relation"]
        if result["predicted_relation"] is not None
        else "__INVALID__"
        for result in predictions
    ]

    accuracy = accuracy_score(y_true, y_pred)

    macro_precision, macro_recall, macro_f1, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=RELATION_LABELS,
            average="macro",
            zero_division=0
        )
    )

    weighted_precision, weighted_recall, weighted_f1, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=RELATION_LABELS,
            average="weighted",
            zero_division=0
        )
    )

    print(f"Accuracy: {accuracy:.4f}")

    print("\nMacro Average")
    print(f"Precision: {macro_precision:.4f}")
    print(f"Recall:    {macro_recall:.4f}")
    print(f"F1:        {macro_f1:.4f}")

    print("\nWeighted Average")
    print(f"Precision: {weighted_precision:.4f}")
    print(f"Recall:    {weighted_recall:.4f}")
    print(f"F1:        {weighted_f1:.4f}")

    print("\nPer-Class Results\n")

    print(
        classification_report(
            y_true,
            y_pred,
            labels=RELATION_LABELS,
            digits=4,
            zero_division=0
        )
    )

In [43]:
# Stage 5.7C - Evaluate Development Set

evaluate_relation_predictions(development_predictions)

Accuracy: 0.5000

Macro Average
Precision: 0.2422
Recall:    0.3216
F1:        0.2582

Weighted Average
Precision: 0.5388
Recall:    0.5000
F1:        0.5171

Per-Class Results

                      precision    recall  f1-score   support

         Association     0.3631    0.3554    0.3592       560
                Bind     0.0851    0.2105    0.1212        19
          Comparison     0.1154    0.6000    0.1935         5
          Conversion     0.0000    0.0000    0.0000         0
         Cotreatment     0.2500    0.4000    0.3077        10
    Drug_Interaction     0.0000    0.0000    0.0000         0
Negative_Correlation     0.3140    0.3750    0.3418       216
Positive_Correlation     0.3047    0.2926    0.2986       352
         No_Relation     0.7478    0.6609    0.7017      1162

           micro avg     0.5052    0.5000    0.5026      2324
           macro avg     0.2422    0.3216    0.2582      2324
        weighted avg     0.5388    0.5000    0.5171      2324



In [45]:
# Stage 5.7D - Inspect Development Relation Confusions

from sklearn.metrics import confusion_matrix
import pandas as pd

y_true = [
    result["gold_relation"]
    for result in development_predictions
]

y_pred = [
    result["predicted_relation"]
    if result["predicted_relation"] is not None
    else "__INVALID__"
    for result in development_predictions
]

valid_eval_labels = RELATION_LABELS + ["__INVALID__"]

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=valid_eval_labels
)

confusion_df = pd.DataFrame(
    cm,
    index=valid_eval_labels,
    columns=valid_eval_labels
)

print("Development Confusion Matrix:")
display(confusion_df)

Development Confusion Matrix:


,Association,Bind,Comparison,Conversion,Cotreatment,Drug_Interaction,Negative_Correlation,Positive_Correlation,No_Relation,__INVALID__
Association,199,19,0,6,1,1,60,122,149,3
Bind,12,4,0,1,0,0,0,1,1,0
Comparison,0,0,3,1,0,0,1,0,0,0
Conversion,0,0,0,0,0,0,0,0,0,0
Cotreatment,0,0,0,0,4,2,1,0,3,0
Drug_Interaction,0,0,0,0,0,0,0,0,0,0
Negative_Correlation,38,4,5,1,7,14,81,10,47,9
Positive_Correlation,139,8,0,3,0,2,31,103,59,7
No_Relation,160,12,18,6,4,3,84,102,768,5
__INVALID__,0,0,0,0,0,0,0,0,0,0


In [46]:
# Show Most Common Development Errors

from collections import Counter

error_pairs = Counter(
    (gold, pred)
    for gold, pred in zip(y_true, y_pred)
    if gold != pred
)

print("Most common prediction errors:\n")

for (gold, pred), count in error_pairs.most_common(15):
    print(f"{gold} -> {pred}: {count}")

Most common prediction errors:

No_Relation -> Association: 160
Association -> No_Relation: 149
Positive_Correlation -> Association: 139
Association -> Positive_Correlation: 122
No_Relation -> Positive_Correlation: 102
No_Relation -> Negative_Correlation: 84
Association -> Negative_Correlation: 60
Positive_Correlation -> No_Relation: 59
Negative_Correlation -> No_Relation: 47
Negative_Correlation -> Association: 38
Positive_Correlation -> Negative_Correlation: 31
Association -> Bind: 19
No_Relation -> Comparison: 18
Negative_Correlation -> Drug_Interaction: 14
Bind -> Association: 12


In [52]:
# Stage 5.8A - Save Frozen Final Prompt Configuration

FINAL_PROMPT_CONFIG_PATH = STAGE_5_DIR / "final_prompt_config.pkl"

final_prompt_config = {
    "model_name": MODEL_NAME,
    "relation_labels": RELATION_LABELS,
    "relation_instructions": RELATION_INSTRUCTIONS,
    "few_shot_selection": FEW_SHOT_SELECTION,
    "num_few_shot_examples": len(few_shot_examples),
    "max_new_tokens": 64,
    "do_sample": False
}

with open(FINAL_PROMPT_CONFIG_PATH, "wb") as f:
    pickle.dump(final_prompt_config, f)

print("Final prompt configuration saved.")
print("Saved to:", FINAL_PROMPT_CONFIG_PATH)

Final prompt configuration saved.
Saved to: /content/drive/MyDrive/Capstone_Project/Stage_5_LLM_RE/final_prompt_config.pkl


In [53]:
# Stage 5.9A - Define Test Checkpoint and Resume Paths

TEST_CHECKPOINT_PATH = STAGE_5_DIR / "test_predictions_checkpoint.pkl"
TEST_FINAL_PATH = STAGE_5_DIR / "test_predictions_final.pkl"

print("Test checkpoint path:")
print(TEST_CHECKPOINT_PATH)

print("\nTest final results path:")
print(TEST_FINAL_PATH)

test_predictions = load_existing_predictions(
    TEST_CHECKPOINT_PATH
)

print(
    "\nPredictions currently available:",
    len(test_predictions)
)

Test checkpoint path:
/content/drive/MyDrive/Capstone_Project/Stage_5_LLM_RE/test_predictions_checkpoint.pkl

Test final results path:
/content/drive/MyDrive/Capstone_Project/Stage_5_LLM_RE/test_predictions_final.pkl
No existing checkpoint found.

Predictions currently available: 0


In [54]:
# Stage 5.9B - Run Full Test Inference with Checkpointing

test_predictions = run_development_inference(
    examples=relation_data["test"],
    checkpoint_path=TEST_CHECKPOINT_PATH,
    existing_predictions=test_predictions,
    max_examples=None,
    checkpoint_every=25
)

Starting from example index: 0
Stopping at example index: 2326
Already completed: 0
1/2326 | Gold: Association | Predicted: Association
2/2326 | Gold: Positive_Correlation | Predicted: Association
3/2326 | Gold: Association | Predicted: Association
4/2326 | Gold: Positive_Correlation | Predicted: No_Relation
5/2326 | Gold: Association | Predicted: Association
6/2326 | Gold: Negative_Correlation | Predicted: Association
7/2326 | Gold: Negative_Correlation | Predicted: Negative_Correlation
8/2326 | Gold: Positive_Correlation | Predicted: Positive_Correlation
9/2326 | Gold: Positive_Correlation | Predicted: Association
10/2326 | Gold: Association | Predicted: Association
11/2326 | Gold: Association | Predicted: Association
12/2326 | Gold: Association | Predicted: Association
13/2326 | Gold: Negative_Correlation | Predicted: Association
14/2326 | Gold: Negative_Correlation | Predicted: Association
15/2326 | Gold: Negative_Correlation | Predicted: None
16/2326 | Gold: Negative_Correlation |

In [55]:
# Stage 5.9C - Save Final Test Predictions

with open(TEST_FINAL_PATH, "wb") as f:
    pickle.dump(test_predictions, f)

print("Final test predictions saved.")
print("Total predictions:", len(test_predictions))
print("Saved to:", TEST_FINAL_PATH)

Final test predictions saved.
Total predictions: 2326
Saved to: /content/drive/MyDrive/Capstone_Project/Stage_5_LLM_RE/test_predictions_final.pkl


In [56]:
# Stage 5.10A - Validate Completed Test Predictions

total_predictions = len(test_predictions)

valid_json_count = sum(
    result["valid_json"]
    for result in test_predictions
)

valid_relation_count = sum(
    result["valid_relation"]
    for result in test_predictions
)

none_prediction_count = sum(
    result["predicted_relation"] is None
    for result in test_predictions
)

print("Total test predictions:", total_predictions)
print("Valid JSON:", f"{valid_json_count}/{total_predictions}")
print("Valid relation labels:", f"{valid_relation_count}/{total_predictions}")
print("Invalid / None predictions:", none_prediction_count)

Total test predictions: 2326
Valid JSON: 2326/2326
Valid relation labels: 2281/2326
Invalid / None predictions: 45


In [57]:
# Stage 5.10B - Evaluate Test Set

evaluate_relation_predictions(test_predictions)

Accuracy: 0.5331

Macro Average
Precision: 0.3405
Recall:    0.6231
F1:        0.3873

Weighted Average
Precision: 0.5621
Recall:    0.5331
F1:        0.5431

Per-Class Results

                      precision    recall  f1-score   support

         Association     0.4095    0.2992    0.3458       635
                Bind     0.0667    0.3333    0.1111         9
          Comparison     0.4000    0.6667    0.5000         6
          Conversion     0.1250    1.0000    0.2222         1
         Cotreatment     0.5500    0.7857    0.6471        14
    Drug_Interaction     0.1176    1.0000    0.2105         2
Negative_Correlation     0.3158    0.4211    0.3609       171
Positive_Correlation     0.3264    0.3877    0.3544       325
         No_Relation     0.7534    0.7145    0.7335      1163

           micro avg     0.5436    0.5331    0.5383      2326
           macro avg     0.3405    0.6231    0.3873      2326
        weighted avg     0.5621    0.5331    0.5431      2326



In [2]:
# Stage 5.11A - Reload Final Qwen Test Predictions

from google.colab import drive
drive.mount("/content/drive")

import pickle
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/Capstone_Project")
STAGE_5_DIR = PROJECT_DIR / "Stage_5_LLM_RE"

TEST_FINAL_PATH = STAGE_5_DIR / "test_predictions_final.pkl"

with open(TEST_FINAL_PATH, "rb") as f:
    test_predictions = pickle.load(f)

print("Loaded Qwen test predictions:", len(test_predictions))

Mounted at /content/drive
Loaded Qwen test predictions: 2326


In [3]:
# Stage 5.11B - Load PubMedBERT RE Final Test Results

import json
from pathlib import Path

STAGE_3_3_DIR = PROJECT_DIR / "Stage_3_3_Outputs"

print("Stage 3.3 folders:")
for path in STAGE_3_3_DIR.iterdir():
    print("-", path.name)

Stage 3.3 folders:
- PubMedBERT_RE_Weighted_Final


In [4]:
# Stage 5.11C - Load PubMedBERT RE Final Test Results

PUBMEDBERT_RE_DIR = (
    STAGE_3_3_DIR / "PubMedBERT_RE_Weighted_Final"
)

PUBMEDBERT_RESULTS_PATH = (
    PUBMEDBERT_RE_DIR / "final_test_results.json"
)

with open(PUBMEDBERT_RESULTS_PATH, "r") as f:
    pubmedbert_test_results = json.load(f)

print("PubMedBERT RE final test results:\n")
print(pubmedbert_test_results)

PubMedBERT RE final test results:

{'model': 'microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract', 'loss_method': 'class_weighted_cross_entropy', 'training_examples': 8340, 'development_examples': 2313, 'test_examples': 2300, 'test_accuracy': 0.6983, 'test_macro_precision': 0.4667, 'test_macro_recall': 0.3539, 'test_macro_f1': 0.3644, 'test_weighted_precision': 0.6955, 'test_weighted_recall': 0.6983, 'test_weighted_f1': 0.6907}


In [7]:
# Stage 5.11D - Restore Relation Labels

RELATION_LABELS = [
    "Association",
    "Bind",
    "Comparison",
    "Conversion",
    "Cotreatment",
    "Drug_Interaction",
    "Negative_Correlation",
    "Positive_Correlation",
    "No_Relation"
]

print("Relation labels restored:", len(RELATION_LABELS))

Relation labels restored: 9


In [8]:
# Stage 5.11E - Compare PubMedBERT and Qwen Relation Extraction Results

import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Recompute Qwen test metrics directly from saved predictions
qwen_y_true = [
    result["gold_relation"]
    for result in test_predictions
]

qwen_y_pred = [
    result["predicted_relation"]
    if result["predicted_relation"] is not None
    else "__INVALID__"
    for result in test_predictions
]

qwen_accuracy = accuracy_score(qwen_y_true, qwen_y_pred)

qwen_macro_precision, qwen_macro_recall, qwen_macro_f1, _ = (
    precision_recall_fscore_support(
        qwen_y_true,
        qwen_y_pred,
        labels=RELATION_LABELS,
        average="macro",
        zero_division=0
    )
)

qwen_weighted_precision, qwen_weighted_recall, qwen_weighted_f1, _ = (
    precision_recall_fscore_support(
        qwen_y_true,
        qwen_y_pred,
        labels=RELATION_LABELS,
        average="weighted",
        zero_division=0
    )
)

comparison_df = pd.DataFrame({
    "Metric": [
        "Test Examples",
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "Weighted Precision",
        "Weighted Recall",
        "Weighted F1"
    ],
    "PubMedBERT": [
        pubmedbert_test_results["test_examples"],
        pubmedbert_test_results["test_accuracy"],
        pubmedbert_test_results["test_macro_precision"],
        pubmedbert_test_results["test_macro_recall"],
        pubmedbert_test_results["test_macro_f1"],
        pubmedbert_test_results["test_weighted_precision"],
        pubmedbert_test_results["test_weighted_recall"],
        pubmedbert_test_results["test_weighted_f1"]
    ],
    "Qwen2.5-7B Few-Shot": [
        len(test_predictions),
        qwen_accuracy,
        qwen_macro_precision,
        qwen_macro_recall,
        qwen_macro_f1,
        qwen_weighted_precision,
        qwen_weighted_recall,
        qwen_weighted_f1
    ]
})

display(comparison_df)

,Metric,PubMedBERT,Qwen2.5-7B Few-Shot
0,Test Examples,2300.0000,2326.000000
1,Accuracy,0.6983,0.533104
2,Macro Precision,0.4667,0.340490
3,Macro Recall,0.3539,0.623134
4,Macro F1,0.3644,0.387275
5,Weighted Precision,0.6955,0.562070
6,Weighted Recall,0.6983,0.533104
7,Weighted F1,0.6907,0.543067


### Relation Extraction Comparison Summary

PubMedBERT achieved stronger overall relation extraction performance, with higher accuracy (**69.83%**) and weighted F1 (**69.07%**) compared with Qwen2.5-7B few-shot extraction (**53.31% accuracy; 54.31% weighted F1**).

Qwen achieved substantially higher macro recall (**62.31% vs 35.39%**) and a slightly higher macro F1 (**38.73% vs 36.44%**), suggesting that the few-shot LLM was more sensitive to minority relation classes but produced more false-positive classifications.

Overall, PubMedBERT provided more reliable performance across the imbalanced BioRED relation dataset, whereas Qwen demonstrated greater recall across relation categories without task-specific fine-tuning.

**Evaluation note:** PubMedBERT was evaluated on **2,300** test examples after truncation-related marker-problem examples were excluded, while Qwen was evaluated on all **2,326** prepared test relation examples.